## Classic speeches 
In this project, I try to assess the complexity level of central bank speeches, focusing on three central banks : the ECB, the Fed and the BoE. To do so, I rely on a dataset of speeches from these three institutions. This dataset comes from this article : Campiglio, E., Deyris, J., Romelli, D. and Scalisi, G., 2025. Warning words in a warming world: Central bank communication and climate change. European Economic Review [Open Access], Vol 178.

In [ ]:
import sys
sys.path.append('/home/onyxia/work/nlp_central_banks/lyna_work')

import module

In [ ]:
# Database of Campiglio, Deyris, Romelli and Scalisi (saved in the SSP Cloud, too heavy for github) :
MY_BUCKET = "lelkamel"
fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"}
)
files_url = f"{MY_BUCKET}/"
fs.get(f"{MY_BUCKET}/","", recursive=True)

# Lecture
specialized_speeches = pd.read_csv("CBS_dataset_v1.0.csv")

#### Checking the data and selecting the subsample of interest

In [ ]:
#I start my analysis in 2000
specialized_speeches["Date"] = pd.to_datetime(specialized_speeches["Date"])
specialized_speeches["Year"] = specialized_speeches["Date"].dt.year
specialized_speeches = specialized_speeches[specialized_speeches["Year"] > 2000]

In [ ]:
# --- check for missing values ---
print("Missing values in each column:")
print(specialized_speeches.isnull().sum())

In [ ]:
# I only select the Fed, the BoE and the ECB
countries_selected = [
    "USA",
    "GBR",
    "ECB"
]
specialized_speeches_filtered = specialized_speeches[
    specialized_speeches["Country"].isin(countries_selected)
]

I make sure there are no errors in the data. Indeed each country is associated with its institution of reference : Brittain is the BoE, the European union is the EU and the USA, the state institutions.

In [ ]:
count_institutions = (
    specialized_speeches_filtered
    .groupby("Country")["CentralBank"]
    .nunique()
)
print(count_institutions)

In [ ]:
us_counts = (
    specialized_speeches_filtered[
        specialized_speeches_filtered["Country"] == "USA"
    ]
    .groupby("CentralBank")
    .size()
    .sort_values(ascending=False)
)

print(us_counts)

### Some descriptive statistics

Do some instutions speak more than others ? Who speaks ?

In [ ]:
df = specialized_speeches_filtered.copy()
country_year = (
    df.groupby(['Year', 'Country'])
      .size()
      .reset_index(name='Count')
)
# Country evolution
plt.figure(figsize=(14, 7))
sns.lineplot(data=country_year, x='Year', y='Count', hue='Country', marker='o', linewidth=2)
plt.grid(True, alpha=0.3)
plt.title('Evolution of Speeches by Country')
plt.xlabel('Year')
plt.ylabel('Number of Speeches')
plt.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


We consider several metrics that aim to capture the **cognitive complexity** of texts—such as sentence length, document length, and grammatical structure.

However, as highlighted in a *June 2025 International Monetary Fund working paper*, these indicators, while informative, **do not fully capture the semantic content of communication**, including the underlying economic rationale, policy intent, or strategic framing. To gain deeper insights, a more advanced NLP analysis is therefore required.

At this stage, we focus on the following metrics:

1. **Average word count by institution**  
   Measures the typical length of documents and provides a proxy for the overall verbosity of communication.

2. **Average sentence length**  
   Captures syntactic complexity at the sentence level, with longer sentences generally indicating higher processing effort.

3. **Flesch–Kincaid score**  
   Evaluates readability and linguistic difficulty based on sentence length and word complexity.

4. **Dependency depth by institution**  
   Reflects grammatical complexity by measuring the depth of syntactic dependency trees.

These metrics provide a useful first layer of analysis for comparing communication styles across institutions, but remain limited to surface-level textual features.

In [ ]:
import numpy as np
from textstat import flesch_reading_ease
import seaborn as sns
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
nltk.download('punkt_tab')

In [ ]:
#Varable to study speech complexity (6 minutes)
df['word_count'] = df['text'].apply(lambda x: len(word_tokenize(x))) 
df['char_count'] = df['text'].apply(len)
df['sentence_count'] = df['text'].apply(lambda x: len(sent_tokenize(x)))
df['avg_word_length'] = df['text'].apply(lambda x: np.mean([len(word) for word in word_tokenize(x)]) if len(word_tokenize(x)) > 0 else 0)
df['avg_sentence_length'] = df['word_count'] / df['sentence_count']
df['flesch_score'] = df['text'].apply(lambda x: flesch_reading_ease(x))

In [ ]:
figures_path = "/home/onyxia/work/nlp_central_banks/lyna_work/figures"
os.makedirs(figures_path, exist_ok=True)

countries = ['ECB', 'GBR', 'USA']
labels = {'ECB': 'ECB', 'GBR': 'BoE', 'USA': 'FED'}

df_plot = df[df['Country'].isin(countries)].copy()
df_plot['Institution'] = df_plot['Country'].map(labels)
df_plot['Year'] = df_plot['Year'].astype(int)

yearly = (
    df_plot
    .groupby(['Institution', 'Year'])
    .agg({
        'word_count': 'mean',
        'avg_sentence_length': 'mean',
        'flesch_score': 'mean',
        'dependency_depth': 'mean'
    })
    .reset_index()
)
metrics_1 = ['word_count', 'avg_sentence_length']
titles_1 = ['Word Count', 'Average Sentence Length']

fig, axs = plt.subplots(2, 2, figsize=(16, 10))

for i, (metric, title) in enumerate(zip(metrics_1, titles_1)):
    sns.boxplot(
        data=df_plot,
        x='Institution',
        y=metric,
        ax=axs[0, i]
    )
    axs[0, i].set_title(f'{title} by Institution')
    axs[0, i].grid(True, alpha=0.3)

    for inst in ['ECB', 'BoE', 'FED']:
        inst_data = yearly[yearly['Institution'] == inst]
        axs[1, i].plot(
            inst_data['Year'],
            inst_data[metric],
            marker='o',
            linewidth=2,
            label=inst
        )

    axs[1, i].set_title(f'{title} Over Time')
    axs[1, i].set_xlabel('Year')
    axs[1, i].set_ylabel(title)
    axs[1, i].grid(True, alpha=0.3)
    axs[1, i].legend()

plt.tight_layout()
plt.savefig(
    f"{figures_path}/speech_complexity_word_sentence_length.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()

In [ ]:
from tqdm import tqdm
import pydantic
import spacy
import thinc

print("pydantic:", pydantic.__version__)
print("spacy:", spacy.__version__)
print("thinc:", thinc.__version__)

nlp = spacy.load("en_core_web_sm")
print("OK")


# -----------------------------
# 3. Function to compute dependency depth of one sentence
# -----------------------------
def sentence_dependency_depth(sent):
    """
    Depth = longest path from sentence root to any terminal node (leaf).
    We count nodes on the path, so a root->child->grandchild path has depth 3.
    """
    root = sent.root

    def depth_from_token(token):
        children = list(token.children)
        if not children:
            return 1
        return 1 + max(depth_from_token(child) for child in children)

    return depth_from_token(root)

# -----------------------------
# 4. Function to compute speech-level dependency depth
# -----------------------------
def speech_dependency_depth(text):
    """
    Compute average sentence dependency depth for one speech.
    You could also use max(sentence depths) instead of mean if preferred.
    """
    doc = nlp(text)

    sentence_depths = []
    for sent in doc.sents:
        # skip extremely short / empty-like sentences
        sent_tokens = [tok for tok in sent if not tok.is_space]
        if len(sent_tokens) == 0:
            continue
        sentence_depths.append(sentence_dependency_depth(sent))

    if len(sentence_depths) == 0:
        return None

    return sum(sentence_depths) / len(sentence_depths)

#df['dependency_depth'] = df['text'].apply(speech_dependency_depth)
# Drop rows where parsing failed
#df = df.dropna(subset=['dependency_depth'])


output_file = "df_with_depth.csv"

# Reprise si fichier existe
if os.path.exists(output_file):
    df_saved = pd.read_csv(output_file)
    df['dependency_depth'] = df_saved['dependency_depth']
else:
    df['dependency_depth'] = None

# Boucle avec barre de progression
for i in tqdm(range(len(df)), desc="Processing speeches"):
    
    if pd.notnull(df.loc[i, 'dependency_depth']):
        continue  # skip déjà fait

    text = df.loc[i, 'text']
    
    try:
        depth = speech_dependency_depth(text)
    except Exception as e:
        depth = None
        print(f"\nError at index {i}: {e}")  # \n pour ne pas casser la barre

    df.loc[i, 'dependency_depth'] = depth

    # 💾 Sauvegarde tous les 10
    if i % 10 == 0:
        df.to_csv(output_file, index=False)

# Sauvegarde finale
df.to_csv(output_file, index=False)
print("Final save complete")

So the speeches seem to be shorter across time (except for the UK). Now, how readable are they ? I start to study leaxical readability metrics (such as the Flesch-Kincaid scores which are widely used over the literature). I also consider a grammatical metric, using dependency parsing (note that I do not use the most developed models, as we are only in a preprocessing step).

The Flesch-Kincaid Ease Score is calculated as:
$$
\text{Score } = 206.835-1.015(\frac{\text{ Total Words }}{\text{ Total Sentences }})-84.6(\frac{\text{ Total Syllables }}{\text{ Total Words }})
$$
The score ranges from 0 to 100, with higher values indicating better readability. In English-language texts, scores above 60 correspond to an 8th-grade reading level, while values between 30 and 50 suggest college-level complexity. Scores below 30 indicate highly technical or specialized content.

In [ ]:
df = pd.read_csv('/home/onyxia/work/nlp_central_banks/lyna_work/speeches_cleaned.csv')

In [ ]:
metrics_2 = ['flesch_score', 'dependency_depth']
titles_2 = ['Flesch/Kincaid Score', 'Dependency Depth']

fig, axs = plt.subplots(2, 2, figsize=(16, 10))

for i, (metric, title) in enumerate(zip(metrics_2, titles_2)):
    sns.boxplot(
        data=df_plot,
        x='Institution',
        y=metric,
        ax=axs[0, i]
    )
    axs[0, i].set_title(f'{title} by Institution')
    axs[0, i].grid(True, alpha=0.3)

    for inst in ['ECB', 'BoE', 'FED']:
        inst_data = yearly[yearly['Institution'] == inst]
        axs[1, i].plot(
            inst_data['Year'],
            inst_data[metric],
            marker='o',
            linewidth=2,
            label=inst
        )

    axs[1, i].set_title(f'{title} Over Time')
    axs[1, i].set_xlabel('Year')
    axs[1, i].set_ylabel(title)
    axs[1, i].grid(True, alpha=0.3)
    axs[1, i].legend()

plt.tight_layout()
plt.savefig(
    f"{figures_path}/speech_complexity_flesch_dependency.png",
    dpi=300,
    bbox_inches='tight'
)
plt.show()

#### Data Preprocessing

In [ ]:
df.info()

In [ ]:
# Filtering the columns (getting Date and speech, country and institution)
df_topic = df.copy()

In [ ]:
selected_banks = [
    "Bank of England",
    "Board of Governors of the Federal Reserve",
    "European Central Bank"
]

df_topic = df_topic[df_topic["CentralBank"].isin(selected_banks)].copy()

In [ ]:
df_topic['Source'].value_counts()

In [ ]:
#Now we do some verifications, to see how speeches are formatted
print(df_topic['Language'].value_counts())
print(pd.crosstab(df_topic['Source'], df_topic['PDF'].notna()))
df_topic['text_length'] = df_topic['text'].str.split().str.len()
print(df_topic.groupby('Source')['text_length'].describe())
mask = df_topic['text_original'].notna()
print(df_topic[mask][['text', 'text_original']].head(2))

In [ ]:
# Les artefacts de footer ECB qu'on a vus — sont-ils présents partout ?
ecb_footer = df_topic['text'].str.contains('CONTACT European Central Bank', na=False)
print(ecb_footer.value_counts())

# Idem pour les artefacts de numéros de page PDF style "1/2001"
page_nums = df_topic['text'].str.contains(r'\d{1,2}/\d{4}\s+\d+', regex=True, na=False)
print(page_nums.value_counts())

In [ ]:
# Footer ECB par CentralBank
print(df_topic.groupby('CentralBank')['text'].apply(
    lambda x: x.str.contains('CONTACT European Central Bank', na=False).mean()
).round(2))

# Numéros de page par Source x PDF
print(df_topic.groupby(['Source', df_topic['PDF'].notna()])['text'].apply(
    lambda x: x.str.contains(r'\d{1,2}/\d{4}\s+\d+', regex=True, na=False).mean()
).round(2))

# Et surtout : quelles banques centrales as-tu ?
print(df_topic['CentralBank'].value_counts())

In [ ]:

# ── 1. Longueur des textes par banque et source ─────────────
df_topic['n_words'] = df_topic['text'].str.split().str.len()

print(df_topic.groupby(['CentralBank', 'Source'])['n_words'].describe().round(0))

# ── 2. Chercher des artefacts fréquents ─────────────────────
patterns = {
    'footer_ECB':     r'CONTACT European Central Bank',
    'footer_BOE':     r'Bank of England.*?©',
    'footer_FED':     r'Federal Reserve.*?Board',
    'page_number':    r'\b\d{1,2}/\d{4}\b',
    'encoding_artefact': r'[A-Za-z]\d[A-Za-z]\d',  # "Rf6ntgen"
    'references':     r'\b(References|Bibliography)\b',
    'tables':         r'(Table \d|Figure \d)',
    'footnotes':      r'^\d{1,2}\.',          # lignes commençant par "1."
    'urls':           r'https?://\S+',
    'brackets_ref':   r'\(\w+,\s*\d{4}\)',    # "(Nickell, 1997)"
    'squarebrackets': r'\[\d+\]',             # "[1]"
}

for name, pat in patterns.items():
    count = df_topic['text'].str.contains(pat, regex=True, na=False).sum()
    pct = count / len(df_topic) * 100
    print(f"{name:25s} : {count:4d} discours ({pct:.1f}%)")

# ── 3. Regarder les textes très courts et très longs ────────
print("\n--- Textes très courts (< 500 mots) ---")
print(df_topic[df_topic['n_words'] < 500][['CentralBank','Source','Year','n_words']].head(10))

print("\n--- Textes très longs (> 15000 mots) ---")
print(df_topic[df_topic['n_words'] > 15000][['CentralBank','Source','Year','n_words']].head(10))

# ── 4. Evolution temporelle ──────────────────────────────────
print("\n--- Discours par année ---")
print(df_topic.groupby('Year')['n_words'].agg(['count','mean']).round(0))

In [ ]:
# 1. C'est quoi exactement le footer FED ?
fed_texts = df_topic[
    df_topic['text'].str.contains('Federal Reserve.*?Board', regex=True, na=False) &
    (df_topic['CentralBank'] == 'Board of Governors of the Federal Reserve')
]['text'].iloc[0]
print(fed_texts[-500:])  # les 500 derniers caractères

# 2. Les square brackets — contexte
sample = df_topic[
    df_topic['text'].str.contains(r'\[\d+\]', regex=True, na=False)
]['text'].iloc[0]
# Trouver un exemple de [1] dans le texte
import re
matches = [(m.start(), m.end()) for m in re.finditer(r'\[\d+\]', sample)]
for start, end in matches[:3]:
    print(repr(sample[max(0,start-100):end+100]))
    print("---")

# 3. Les textes très courts — sont-ils utilisables ?
short = df_topic[df_topic['n_words'] < 500]
print(short['text'].iloc[0][:500])

# 4. Un texte très long BoE — est-ce un paper académique ?
long_boe = df_topic[df_topic['n_words'] > 15000].iloc[0]
print(long_boe['text'][:300])
print("...")
print(long_boe['text'][-300:])

Now, I select the speeches that look the most alike. For example, we can see that state central bank speeches in the US are on average shorter than federal speeches or than that of the ECB and the BOE. Thus, for the US, we will only focus on the Fed speeches.

In [ ]:
boe_speech = df_topic[
    df_topic["CentralBank"] == "Bank of England"
].iloc[0]["text"]

print(boe_speech)

In [ ]:
fed_speech = df_topic[
    df_topic["CentralBank"] == "Board of Governors of the Federal Reserve"
].iloc[0]["text"]

print(fed_speech)

In [ ]:
pd.set_option('display.max_colwidth', None)
print(df_topic.loc[0, "text"])

In [ ]:
# Application
df_topic['text_clean'] = df_topic.apply(clean_text, axis=1)
df_topic['n_words_clean'] = df_topic['text_clean'].str.split().str.len()

diff = df_topic['n_words'] - df_topic['n_words_clean']

print(diff.describe().round(1))
print(f"\nDiscours avec > 200 mots supprimés  : {(diff > 200).sum()}")
print(f"Discours avec > 500 mots supprimés  : {(diff > 500).sum()}")
print(f"Discours avec > 1000 mots supprimés : {(diff > 1000).sum()}")
print(f"Discours avec > 2000 mots supprimés : {(diff > 2000).sum()}")

# Vérification sur les cas problématiques
print("\n--- Cas cibles ---")
for idx in [4871, 2532, 2512, 4167, 4924]:
    orig  = df_topic.loc[idx, 'text']
    clean = df_topic.loc[idx, 'text_clean']
    print(f"Index {idx} | {df_topic.loc[idx, 'Year']} | {len(orig.split())} -> {len(clean.split())} mots")

In [ ]:
worst = df_topic.assign(diff=df_topic['n_words'] - df_topic['n_words_clean'])
worst_12 = worst.nlargest(12, 'diff')[['CentralBank', 'Source', 'Year', 'n_words', 'n_words_clean', 'diff']]
print(worst_12)

# Pour chacun, montrer le contexte de coupure
for idx in worst_12.index:
    orig  = df_topic.loc[idx, 'text']
    clean = df_topic.loc[idx, 'text_clean']
    cut   = len(clean)
    print(f"\nIndex {idx} | {df_topic.loc[idx, 'CentralBank'][:3]} | {df_topic.loc[idx, 'Year']}")
    print(repr(orig[max(0, cut-100):cut+100]))

Récapitulatif du preprocessing accompli jusqu'à présent : 
1) Supression propre des sections Rferences/Bibliography (papers académiques de la BoE ou de la Fed)
2) Footer ECB supprimés
3) Notes de bas de page [1] supprimés
4) Numéros de page PDF supprimés
5) Artefacts d'encoding corrigés
6) Apostrophes Fed corrigées

Now, we have to select only the texts that are alike. So that Bertopic performs well.

In [ ]:
# Distribution des rôles
print(df_topic['Role'].value_counts())

# Longueur moyenne par rôle
print(df_topic.groupby('Role')['n_words'].agg(['count', 'mean', 'std']).round(0).sort_values('count', ascending=False))

# Croisement rôle x banque
print(pd.crosstab(df_topic['Role'], df_topic['CentralBank']))

In [ ]:
 Qui sont les "Board members" de la Fed ?
print("=== FED Board members - exemples ===")
print(df_topic[
    (df_topic['CentralBank'] == 'Board of Governors of the Federal Reserve') &
    (df_topic['Role'] == 'Board member')
]['Authorname'].value_counts().head(10))

# Qui sont les "Governors" par banque ?
print("\n=== Governors par banque ===")
print(df_topic[df_topic['Role'] == 'Governor'].groupby(
    ['CentralBank', 'Authorname']
)['Role'].count().reset_index().sort_values(['CentralBank', 'Role'], ascending=[True, False]))

# Les Deputy Governors - quelle banque ?
print("\n=== Deputy Governors par banque ===")
print(pd.crosstab(df_topic[df_topic['Role'] == 'Deputy Governor']['CentralBank'],
                  df_topic[df_topic['Role'] == 'Deputy Governor']['Role']))

# Longueur par rôle ET banque
print("\n=== Longueur moyenne par rôle x banque ===")
print(df_topic.groupby(['CentralBank', 'Role'])['n_words'].agg(['count','mean']).round(0))

In [ ]:
df_filtered = df_topic[df_topic['Role'].isin(['Governor', 'Deputy Governor'])].copy()

print(f"Corpus original  : {len(df_topic)} discours")
print(f"Corpus filtré    : {len(df_filtered)} discours")
print(f"\nPar banque :")
print(df_filtered.groupby(['CentralBank', 'Role'])['Authorname'].count())
print(f"\nLongueur moyenne après filtre :")
print(df_filtered.groupby('CentralBank')['n_words'].agg(['count','mean','std']).round(0))

# Est-ce que les textes très longs disparaissent ?
print(f"\nTextes > 10000 mots avant filtre : {(df_topic['n_words'] > 10000).sum()}")
print(f"Textes > 10000 mots après filtre : {(df_filtered['n_words'] > 10000).sum()}")